# Sampling wave fields from a trained diffusion model, conditioned on wind forcing

This notebook loads a checkpoint from the `OPTION1_periodic` conditional diffusion model
(trained by [`example/meandiff_periodic.py`](meandiff_periodic.py)) and uses it to sample
bulk wave variables — significant wave height (Hs), mean period (Tm), mean direction (θm) —
conditioned on 10&nbsp;m wind forcing and a land/ice mask.

**Model summary**
- Architecture: `myUnet`, a longitude-periodic U-Net (`wavediffusion.model_unet.myUnet`), wrapped with `Scaled(...)` so it predicts noise `eps` directly (`wavediffusion.model.Scaled`)
- Predicted fields (`in_ch=3`): `[hs, t0m1, dir]` (log1p applied to `hs` before normalization)
- Conditioning fields (`precond_ch=14`): `[uwnd, vwnd, dpt, ice_frac]` at the current step, plus 5 daily historical `(uwnd, vwnd)` snapshots (1–5 days back)
- Domain: global grid resized to 320×320, periodic convolutions along longitude
- Checkpoint: `/pscratch/sd/j/jiarongw/final/OPTION1_periodic/ckpt_6.pt` (epoch 6)

Run this with the **waveml** kernel. `torch.cuda.is_available()` will be `False` on a login
node — the 40-step reverse-diffusion sampler is slow on CPU, so for real use grab a GPU node
first, e.g.:
```
salloc --nodes=1 --qos=interactive --time=01:00:00 --constraint=gpu --gpus=1 --account=m4874
```

In [ ]:
import sys, os
sys.path.insert(0, '/global/homes/j/jiarongw/wavediffusion/src')

# wavediffusion.diffusion imports accelerate, which (as of this checkpoint's env) needs
# huggingface_hub -- install it once into user site-packages if the import below fails.
try:
    from accelerate import Accelerator
except ModuleNotFoundError as e:
    if 'huggingface_hub' not in str(e):
        raise
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--user', 'huggingface_hub'], check=True)
    from accelerate import Accelerator

import numpy as np
import torch
import torchvision.transforms as tf
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from wavediffusion.model_unet import myUnet
from wavediffusion.model import Scaled
from wavediffusion.wavedata import npyDataWndHist
from wavediffusion.diffusion import ScheduleLogLinear, samples
from wavediffusion.waveutils import plot_sample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. Load the checkpoint and rebuild the model

The normalization statistics (`meanx`/`stdx` for the predicted fields, `meanf`/`stdf` for the
forcing) are saved as buffers inside the checkpoint itself, so we read them straight from
`ckpt['model']` rather than depending on a separate stats file staying in sync.

In [ ]:
CKPT_PATH = '/pscratch/sd/j/jiarongw/final/OPTION1_periodic/ckpt_6.pt'

ckpt = torch.load(CKPT_PATH, map_location='cpu')
print('epoch:', ckpt['epoch'])

state_dict = ckpt['model']
meanx, stdx = state_dict['meanx'], state_dict['stdx']
meanf, stdf = state_dict['meanf'], state_dict['stdf']

model = Scaled(myUnet)(
    in_dim=320, in_ch=3, out_ch=3, ch=256, precond_ch=14,
    scale=(meanx, stdx, meanf, stdf),
    ch_mult=(1, 2, 2), attn_resolutions=(16,), periodic=True,
)
model.load_state_dict(state_dict)
model.to(device).eval()
print(f'loaded model with {sum(p.numel() for p in model.parameters()):,} parameters')

Optional: the checkpoint also stores an EMA (exponential moving average) copy of the weights,
which is what `sample_and_save` uses during training and generally gives smoother samples.
Run this cell to swap the model's weights for the EMA weights in place.

In [ ]:
from torch_ema import ExponentialMovingAverage as EMA

ema = EMA(model.parameters(), decay=0.999)
ema.load_state_dict(ckpt['ema'])
ema.copy_to(model.parameters())  # overwrite model weights in-place with the EMA average

## 2. Prepare a wind-forcing input\n\nTo sample, the model needs:\n- `cond`: a `(B, 14, 320, 320)` tensor — `[uwnd, vwnd, dpt, ice_frac]` plus 5 daily historical\n  `(uwnd, vwnd)` snapshots, normalized with `meanf`/`stdf` above\n- `mask`: a `(B, 1, 320, 320)` land/ice mask, so masked-out pixels stay at zero throughout\n  sampling\n\nThe simplest way to get both correctly formatted is to reuse `npyDataWndHist`, which handles\nthe resizing to 320×320, the `log1p` + normalization, and the mask. The example below points it\nat one of the training pipeline's held-out months; swap in your own forcing `.npy` file to\nsample under a different wind scenario — see the **Appendix** at the end of this notebook for\nthe full data format (exact channels, units, history construction, normalization).

In [ ]:
DATA_PATH = '/global/homes/j/jiarongw/scratch_folder/wave_data/mean_global/'
OPTION = 1

example = npyDataWndHist(
    [(os.path.join(DATA_PATH, 'wavemean_201904.npy'), os.path.join(DATA_PATH, 'forcing_201904.npy'))],
    resize_x=(320, 320), resize_f=(320, 320),
    landmaskname=os.path.join(DATA_PATH, 'mask.npy'),
    use_icymask=True, compute_stats=False,
    meanx=meanx, stdx=stdx, meanf=meanf[:4], stdf=stdf[:4],  # base 4 channels only -- the dataset re-expands to 14 by appending 5 copies of meanf[:2]/stdf[:2] internally
    OPTION=OPTION,
)

loader = DataLoader(example, batch_size=4, shuffle=True)
x_true, f, mask = next(iter(loader))
x_true, f, mask = x_true.to(device), f.to(device), mask.to(device)
print('x_true (ground truth, normalized):', x_true.shape)  # (B, 3, 320, 320): hs, t0m1, dir
print('f (wind forcing incl. 5-day history, normalized):', f.shape)  # (B, 14, 320, 320)
print('mask (land/ice):', mask.shape)  # (B, 1, 320, 320)

## 3. Sample

`samples()` implements a general SDE/ODE sampler (`gam`, `mu` interpolate between DDPM- and
DDIM-style trajectories). `gam=1, mu=0.5` reproduces the DDPM sampling used at training time.
It yields the intermediate `x_t` at every reverse-diffusion step (re-masked so land/ice stays
zero); we only keep the final denoised sample `x0`.

In [ ]:
schedule = ScheduleLogLinear(sigma_min=0.01, sigma_max=100, N=80)
sigmas = schedule.sample_sigmas(40)  # 40 reverse-diffusion steps, same as training-time sampling

n_samples = f.shape[0]  # one sample per forcing snapshot in the batch

with torch.no_grad():
    *_, x0 = samples(
        model, sigmas, gam=1, mu=0.5, batchsize=n_samples,
        cond=f, mask=mask,
    )
print('sampled x0:', x0.shape)

### Ensemble: draw several samples for one forcing snapshot

Since sampling starts from random noise, repeating a single forcing/mask pair across the batch
dimension draws an ensemble of plausible wave fields for the *same* wind forcing — useful for
looking at sample-to-sample variability or computing an ensemble mean.

In [ ]:
n_ens = 4
i = 0  # index of the forcing snapshot (within the batch above) to condition the ensemble on
cond_i = f[[i]].repeat(n_ens, 1, 1, 1)
mask_i = mask[[i]].repeat(n_ens, 1, 1, 1)

with torch.no_grad():
    *_, x0_ens = samples(
        model, sigmas, gam=1, mu=0.5, batchsize=n_ens,
        cond=cond_i, mask=mask_i,
    )
print('ensemble of samples for one forcing snapshot:', x0_ens.shape)  # (n_ens, 3, 320, 320)

## 4. Un-normalize and visualize

`example.invert_x` / `example.invert_f` undo the normalization and the `log1p` on `hs`, and
resize back to the original 320×720 grid. `plot_sample` renders truth vs. samples using the
same conventions as the rest of this repo (`Blues` for Hs, `pink_r` for Tm, `twilight` for θm
— see `analysis/paper_figure/figure2.ipynb`).

In [ ]:
full_mask = tf.Resize((example.original_H, example.original_W))(mask[i].cpu())

x_true_i = example.invert_x(x_true[i].cpu()) * full_mask
x_ens_phys = torch.stack([example.invert_x(s.cpu()) * full_mask for s in x0_ens])
f_i = example.invert_f(f[i].cpu())

fig = plot_sample(
    x_true_i.numpy()[:, ::-1],       # flip latitude for north-up display
    x_ens_phys.numpy()[:, :, ::-1],
    f_i.numpy()[:, ::-1],
    OPTION=OPTION,
)

The forcing tensor's first 4 channels (before the wind-history channels) are, in order,
`[uwnd, vwnd, dpt, ice_frac]` — note this differs from `waveutils.plot_forcing`, which assumes
an older 3-channel `[uwnd, vwnd, ice_frac]` layout, so we plot it directly here instead.

In [ ]:
titles = ['wind u (m/s)', 'wind v (m/s)', 'depth (m)', 'ice fraction']
cmaps = ['PiYG', 'PiYG', 'viridis', 'Purples']
fig, axes = plt.subplots(1, 4, figsize=(20, 3), dpi=100)
for j, (t, c) in enumerate(zip(titles, cmaps)):
    im = axes[j].imshow(f_i.numpy()[j][::-1], cmap=c)
    axes[j].set_title(t); axes[j].set_xticks([]); axes[j].set_yticks([])
    fig.colorbar(im, ax=axes[j], fraction=0.046, pad=0.04)
plt.tight_layout()

## 5. Quick check: one-step (conditional-mean) sampling

For a fast sanity check without running the full 40-step reverse diffusion (handy on a CPU-only
login node), evaluate the model's one-step denoised estimate at a fixed high noise level — this
is what `sample_and_save` logs as `*_onestep_*.png` during training. It is a blurrier,
lower-variance point estimate, not a proper posterior sample.

In [ ]:
sigma0 = sigmas[0].to(device)
with torch.no_grad():
    xt = model.rand_input(n_samples).to(device) * sigma0 * mask
    eps = model.predict_eps_cfg(xt, sigma0, cond=f, cfg_scale=0) * mask
    x0_onestep = xt - eps * sigma0
print('one-step estimate:', x0_onestep.shape)

## Appendix: forcing & target `.npy` data format\n\nReference for building your own conditioning data, based on `wavediffusion.wavedata.npyDataWndHist`\nand the example files in `DATA_PATH` above (`forcing_201904.npy` / `wavemean_201904.npy` / `mask.npy`).\n\n### Raw file layout\n\n| file | shape | dtype | axes |\n|---|---|---|---|\n| `forcing_YYYYMM.npy` (`F`) | `(T, 5, 320, 720)` | float | `(time, channel, lat, lon)` |\n| `wavemean_YYYYMM.npy` (`X`) | `(T, 9, 320, 720)` | float32 | `(time, channel, lat, lon)` |\n| `mask.npy` | `(320, 720)` | int | `(lat, lon)`, static (no time axis) |\n\n- Grid: 320×720 (0.5° global grid), longitude periodic — `X`/`F` are resized to 320×320\n  before being fed to the model (`resize_x`/`resize_f=(320,320)`) and resized back to 320×720\n  by `invert_x`/`invert_f`. Keep the same row/column convention (pole at row 0, etc.) as these\n  example files — nothing downstream re-derives orientation from coordinates.\n- Time cadence: 3-hourly, `SAMPLES_PER_DAY = 8` (hardcoded in `npyDataWndHist`); `T=240` for a\n  30-day month. `X[t]` and `F[t]` must be the same timestamp.\n- Missing/invalid values are NaN; `FillNaN(0.0)` runs *before* normalization, so NaNs become 0\n  post-normalization, not `meanf`/`meanx`.\n\n### Forcing channels (`F`, 5 raw channels)\n\n| idx | name | units | range (example file) | notes |\n|---|---|---|---|---|\n| 0 | `uwnd` | m/s | −27.5 to 25 | zonal 10 m wind |\n| 1 | `vwnd` | m/s | similar to `uwnd` | meridional 10 m wind |\n| 2 | `icymask` | {0, 1} | — | 1 = usable open-ocean pixel *at this timestep* (0 over land and sea ice); **not** fed to the network — this is exactly the dynamic `mask` returned by the dataset and passed to `samples(..., mask=...)` above |\n| 3 | `dpt` | m | 1 to 8620.5 | bathymetric depth |\n| 4 | `ice_frac` | fraction in [0, 1], else NaN | — | sea-ice concentration; NaN where no ice/no estimate (filled to 0 before normalization) |\n\nThe network's `cond` only uses channels `[0, 1, 3, 4]` (`uwnd, vwnd, dpt, ice_frac`) — channel 2\n(`icymask`) is deliberately skipped from `cond` and surfaces instead as the dataset's third\nreturn value, `mask`. The **static** land mask (`mask.npy`, 1 = ocean, 0 = land) is separate\nfrom this per-timestep `icymask`, and is baked automatically into `tf_x`/`tf_f` — you never\napply it yourself.\n\n### Wind history channels (appended after the 4 base channels)\n\nFor `hist_duration_days=D`, `hist_freq_days=k` (`D` must be a multiple of `k`), `npyDataWndHist`\nappends `n_hist = D/k` historical `(uwnd, vwnd)` snapshots, `k, 2k, ..., D` days before the\ncurrent step, each pulled from the *same* `F` file's raw channels 0/1 at earlier time indices\n(`SAMPLES_PER_DAY=8` rows/day). Total conditioning channels = `4 + 2*n_hist`.\n\nThe `OPTION1_periodic` checkpoint (`precond_ch=14`) was trained with the defaults\n`hist_duration_days=5, hist_freq_days=1` — i.e. `n_hist=5`, wind from 1, 2, 3, 4, 5 days ago,\ndaily — giving the exact 14-channel order used throughout this notebook:\n\n```\n[uwnd_t, vwnd_t, dpt_t, ice_frac_t,\n uwnd_{t-1d}, vwnd_{t-1d}, uwnd_{t-2d}, vwnd_{t-2d}, uwnd_{t-3d}, vwnd_{t-3d},\n uwnd_{t-4d}, vwnd_{t-4d}, uwnd_{t-5d}, vwnd_{t-5d}]\n```\n\nBecause history reaches `hist_duration_days` back, the dataset trims that many samples off the\nfront of the file list (`__len__` subtracts `padding = hist_duration_days * SAMPLES_PER_DAY`) —\na custom scenario needs at least 5 days of lead-in wind before the first timestep you actually\nwant to sample.\n\n### Target channels (`X`, 9 raw channels — only needed for the ground-truth comparison panel)\n\n`hs, t0m1, dir, spr, uuss, vuss, mssu, mssc, mssd` (significant wave height, mean period, mean\ndirection, directional spread, Stokes drift u/v, and three mean-square-slope components). This\n(`OPTION=1`) checkpoint only predicts/compares `[hs, t0m1, dir]` (indices 0,1,2); `hs` gets\n`log1p`'d before normalization (undone by `expm1` inside `invert_x`). The other 6 channels exist\nfor other `OPTION`s in this repo and are irrelevant here — if you have no ground truth, any\nplaceholder in those first 3 channels works, since `x_true` is only used for the comparison plot,\nnever fed to the model.\n\n### Normalization\n\nEach forcing/target channel is normalized as `(value - mean[c]) / std[c]`, with `meanx/stdx`\n(3-channel) and `meanf/stdf` (14-channel, already including the repeated history-channel stats)\nread straight from the checkpoint as `state_dict['meanx']` etc. `ice_frac`'s mean/std are fixed\nto `(0, 1)` (identity) rather than fit from data.\n\n**Gotcha hit while building this notebook:** `npyDataWndHist.__init__` takes the *base* 4-channel\n`meanf`/`stdf` and re-expands them to 14 itself (by appending `n_hist` copies of `meanf[:2]`); if\nyou hand it the checkpoint's already-14-channel `meanf`/`stdf` directly it silently builds a\n24-channel normalizer and crashes with a shape mismatch against the real 14-channel tensor. That's\nwhy the forcing cell above passes `meanf[:4], stdf[:4]`. If instead you normalize a hand-built\n14-channel tensor yourself (bypassing the dataset class, see below), use the full checkpoint\n`meanf`/`stdf` directly, channel-for-channel — the history entries are just repeats of\n`meanf[0:2]`/`stdf[0:2]` anyway, so both approaches agree numerically.\n\n### Using your own wind forcing\n\nEither:\n\n1. **Match the `.npy` layout above** and reuse the cells in this notebook: an `F` array shaped\n   `(T, 5, H, W)` with channels `[uwnd, vwnd, icymask, dpt, ice_frac]` at 3-hourly cadence, and an\n   `X` array shaped `(T, 9, H, W)` (placeholder is fine if you have no ground truth) — point\n   `npyDataWndHist` at your own files in place of `wavemean_201904`/`forcing_201904` above.\n\n2. **Build `cond`/`mask` tensors directly**, bypassing the dataset class: assemble a\n   `(B, 14, H, W)` tensor in the channel order given above, in physical units, resize to\n   320×320, normalize with `(f_phys - meanf) / stdf` (the full 14-channel checkpoint stats, per\n   the gotcha above), and pass it as `cond` to `samples(...)`. `mask` should be 1 over open ocean\n   and 0 over land/ice at the same resolution.